Name: Mia Parker

Github Username: emeliap-usc

ID: 1412-4907-44

In [1]:
import pandas as pd
import numpy as np
import os
import random

## 1c)

### i

We can use min, max, mean, median, standard deviation, quartile 1, and quartile 3.

### ii

In [ ]:
start_path = "../data/HW3 Data Unsorted"

features=[]

for activity in os.listdir(start_path):
    activity_path = os.path.join(start_path, activity)

    if not os.path.isdir(activity_path):
        continue

    for data in os.listdir(activity_path):

        data_path = os.path.join(activity_path, data)

        if not data_path.endswith(".csv"):
            continue
        
        columns = [
            "time",
            "avg_rss12",
            "var_rss12",
            "avg_rss13",
            "var_rss13",
            "avg_rss23",
            "var_rss23"]

        with open(data_path, "r") as f:
            for line in f:
                if line.strip() and not line.startswith("#"):
                    first_data_line = line.strip()
                    break

        if "," in first_data_line:
            df = pd.read_csv(
                data_path,
                comment="#",
                names=columns,
                usecols=range(7)
            )
        else: #all because of bending2 dataset4!
            df = pd.read_csv(
                data_path,
                comment="#",
                names=columns,
                sep=r"\s+"
            )

        df = df.drop(columns=["time"])

        row = {
            "activity": activity,
            "filename": data_path
        }

        # Calculate features for each of the 6 time series
        count = 1
        for column in df.columns:


            row[f"min_{count}"] = df[column].min()
            row[f"max_{count}"] = df[column].max()
            row[f"mean_{count}"] = df[column].mean()
            row[f"median_{count}"] = df[column].median()
            row[f"std_{count}"] = df[column].std()
            row[f"q1_{count}"] = df[column].quantile(0.25)
            row[f"q3_{count}"] = df[column].quantile(0.75)
            count+=1

        features.append(row)

features_df = pd.DataFrame(features)



In [3]:
features_df

,activity,filename,min_1,max_1,mean_1,median_1,std_1,q1_1,q3_1,min_2,...,std_5,q1_5,q3_5,min_6,max_6,mean_6,median_6,std_6,q1_6,q3_6
0,bending1,/Users/miaparker/Desktop/USC/DSCI 552/HW3 Data...,36.25,48.00,43.969125,44.50,1.618364,43.3100,44.67,0.0,...,3.318301,20.5000,23.7500,0.0,2.96,0.555312,0.490,0.487826,0.0000,0.8300
1,bending1,/Users/miaparker/Desktop/USC/DSCI 552/HW3 Data...,37.00,48.00,43.454958,43.25,1.386098,42.5000,45.00,0.0,...,2.488862,22.2500,24.0000,0.0,5.26,0.679646,0.500,0.622534,0.4300,0.8700
2,bending1,/Users/miaparker/Desktop/USC/DSCI 552/HW3 Data...,33.00,47.75,42.179812,43.50,3.670666,39.1500,45.00,0.0,...,3.849448,30.4575,36.3300,0.0,2.18,0.613521,0.500,0.524317,0.0000,1.0000
3,bending1,/Users/miaparker/Desktop/USC/DSCI 552/HW3 Data...,33.00,45.75,41.678063,41.75,2.243490,41.3300,42.75,0.0,...,2.411026,28.4575,31.2500,0.0,1.79,0.383292,0.430,0.389164,0.0000,0.5000
4,bending1,/Users/miaparker/Desktop/USC/DSCI 552/HW3 Data...,37.25,45.00,40.624792,40.50,1.476967,39.2500,42.00,0.0,...,2.188449,33.0000,36.0000,0.0,1.92,0.570583,0.430,0.582915,0.0000,1.3000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83,cycling,/Users/miaparker/Desktop/USC/DSCI 552/HW3 Data...,18.50,44.25,35.752354,36.00,4.614802,33.0000,39.33,0.0,...,3.120057,14.0000,18.0625,0.0,9.39,3.069667,2.770,1.748326,1.7975,4.0600
84,cycling,/Users/miaparker/Desktop/USC/DSCI 552/HW3 Data...,24.25,45.00,37.177042,36.25,3.581301,34.5000,40.25,0.0,...,2.890347,17.9500,21.7500,0.0,9.34,2.921729,2.500,1.852600,1.5000,3.9000
85,cycling,/Users/miaparker/Desktop/USC/DSCI 552/HW3 Data...,23.33,43.50,36.244083,36.75,3.822016,33.4575,39.25,0.0,...,3.617702,15.7500,21.0000,0.0,11.15,3.530500,3.110,1.963685,2.1700,4.6175
86,cycling,/Users/miaparker/Desktop/USC/DSCI 552/HW3 Data...,26.25,44.25,36.957458,36.29,3.434863,34.5000,40.25,0.0,...,2.851673,14.0000,18.2500,0.0,8.34,2.934625,2.525,1.631380,1.6600,4.0300


### iii)

In [5]:
feature_columns = [
    col for col in features_df.columns
    if col.startswith(("min_", "max_", "mean_", "median_", "std_", "q1_", "q3_"))
]

random.seed(42)
n = 10000


results = []

for feature in feature_columns:
    
    values = features_df[feature].to_numpy()
    original_std = np.std(values, ddof=1)
    
    bootstrap_stds = []
    
    for i in range(n):
        sample = random.choices(values, k=len(values))
        bootstrap_stds.append(np.std(sample, ddof=1))
    
    # 90% percentile bootstrap confidence interval
    lower = np.percentile(bootstrap_stds, 5)
    upper = np.percentile(bootstrap_stds, 95)
    
    results.append({
        "feature": feature,
        "org_std": original_std,
        "90% CI lower": lower,
        "90% CI upper": upper
    })

std_results = pd.DataFrame(results)

print(std_results)

     feature   org_std  90% CI lower  90% CI upper
0      min_1  9.569975      8.265231     10.778773
1      max_1  4.394362      3.332634      5.306268
2     mean_1  5.335718      4.704508      5.888251
3   median_1  5.440054      4.796165      5.990619
4      std_1  1.772153      1.572190      1.946934
5       q1_1  6.153590      5.567069      6.651897
6       q3_1  5.138925      4.349794      5.845663
7      min_2  0.000000      0.000000      0.000000
8      max_2  5.062729      4.623746      5.405873
9     mean_2  1.574164      1.398985      1.706339
10  median_2  1.412244      1.235354      1.540173
11     std_2  0.884105      0.801826      0.942330
12      q1_2  0.946386      0.830326      1.034302
13      q3_2  2.125266      1.891583      2.296728
14     min_3  2.956462      2.762278      3.105987
15     max_3  4.875137      4.160757      5.450639
16    mean_3  4.008380      3.424055      4.494119
17  median_3  4.036396      3.416281      4.538333
18     std_3  0.946710      0.7

I would select the median, first quartile, and third quartile as my three preferred time-domain features. The median provides a measure of the center of a distribution that is less sensitive to extreme values than the mean. The first and third quartiles describe the lower and upper portions of the distribution, and together capture the middle 50% of observations. However, because the three features are related, they may contain some overlapping information.

## 2.

a) Because the cubic model includes the linear model, it can always do at least as well as the linear. So, the linear model's RSS will be the same or smaller than the linear model's RSS.

b) Because the true relationship is linear, it is likely that the linear model will perform better on the test data. The cubic model might perform better on the training data, but because it is actually linear, that would only be from overfitting the data. So the linear model will likely have lower RSS on the test data.

c) Similarly to part a, the cubic model would likely perform better than the linear model on the training data. Whether the true relationship is linear or cubic or other, the cubic would perform better as it fits the data stronger. In the end this doesnt matter, as it is what performs best on the testing data that shows the best model.

d) Without actually creating models and looking at the RSS, we cannot determine which would actually perform better on the testing data. If the data is slightly non-linear, it could still be closer to a linear relationship than a cubic, so it would have the lower RSS even though its not actually linear. If it is closer to a cubic relationship, of course, the cubic model would have the lower RSS. We are unable to say fully which would perform better.